# Stage 8 — Check the published adapter

Downloads the adapter **from the Hugging Face Hub** and checks that it is the model that was
evaluated (decision D19):

1. the weights on the Hub have the same sha256 as the ones saved in Stage 5;
2. the **usage example of the model card** (`model_card/usage_example.py`), run as is, reproduces
   the Stage 5 result on the 500 **selection** reviews of dev.

**Pass criterion (fixed before running):** macro-F1 within ±0.010 of 0.8994 (Stage 5) **and** at
least 99% of the predictions equal to the saved ones (`predictions/dev_select/smollm3_qlora_human.csv`).
A few flips are expected: the card scores one review at a time, Stage 5 scored padded batches,
and fp16 results depend slightly on the computation path.

**The test set is not used.**

While the Hub repository is private, the notebook needs your Hugging Face token as a secret named
`HF_TOKEN` (read access is enough):

- **Kaggle:** *Add-ons → Secrets → Add a new secret*, label `HF_TOKEN`, then tick it for this
  notebook. Settings: *Accelerator → GPU T4 x2*, *Internet → on*; *Save Version → Save & Run All*.
- **Colab:** the key icon (*Secrets*) on the left, name `HF_TOKEN`, enable notebook access;
  *Runtime → Change runtime type → T4 GPU*, then *Run all*.

Once the repository is public, the secret is no longer needed.

In [ ]:
import importlib.util
import os
import subprocess
import sys

import torch

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    PLATFORM, WORK_DIR, OUTPUT_DIR = "kaggle", "/tmp", "/kaggle/working"
elif importlib.util.find_spec("google.colab") is not None:
    PLATFORM, WORK_DIR, OUTPUT_DIR = "colab", "/content", "/content"
else:
    raise RuntimeError("Run this notebook on Google Colab or Kaggle")
print("platform:", PLATFORM)

assert torch.cuda.is_available(), (
    "No GPU. Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2"
)
GPU_NAME = torch.cuda.get_device_name(0)
print(GPU_NAME)

# The token goes into the environment, where huggingface_hub reads it; it is never printed.
try:
    if PLATFORM == "kaggle":
        from kaggle_secrets import UserSecretsClient

        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    else:
        from google.colab import userdata

        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN secret loaded")
except Exception as error:  # a public repository needs no token
    print("no HF_TOKEN secret:", type(error).__name__)

REPO_URL = "https://github.com/PedroRicci12/ptbr-review-lora.git"
REPO_DIR = f"{WORK_DIR}/ptbr-review-lora"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")
REPO_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("repository commit:", REPO_COMMIT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "bitsandbytes"], check=True)

## 1. Data

Same script as on the laptop; the cell stops if the counts differ from the repository.

In [ ]:
subprocess.run(
    [sys.executable, "scripts/prepare_data.py"],
    check=True, stdout=subprocess.DEVNULL, env={**os.environ, "PYTHONPATH": "src"},
)
diff = subprocess.run(["git", "diff", "--exit-code", "reports/data_summary.json"], capture_output=True, text=True)
assert diff.returncode == 0, "Prepared data differs from the repository:\n" + diff.stdout
print("data_summary.json identical to the repository")

import pandas as pd

from ptbr_review_lora.data import model_input, split_dev

dev = pd.read_parquet("data/processed/repro_dev.parquet")
_, select = split_dev(dev, train_size=1500, seed=42)
select_texts = model_input(select, use_title=True).tolist()
y_select = select["label"].tolist()
print("selection rows:", len(select))

## 2. The weights on the Hub

`hf_hub_download` fetches the file the same way `PeftModel.from_pretrained` does (same cache).

In [ ]:
import hashlib
from pathlib import Path

from huggingface_hub import HfApi, hf_hub_download

ADAPTER_REPO = "PedroRicci12/smollm3-3b-ptbr-review-sentiment-qlora"
EXPECTED_SHA256 = "ae8904dfd3770e5c1452990fbf6af9e18fb83307d58b824abafbb987ca6389a1"  # Stage 5 zip
REFERENCE_F1 = 0.8994076276155476   # reports/smollm3_qlora_human_select.json
F1_TOLERANCE = 0.010
MIN_AGREEMENT = 0.99

hub_commit = HfApi().model_info(ADAPTER_REPO).sha
weights = hf_hub_download(ADAPTER_REPO, "adapter_model.safetensors")
hub_sha256 = hashlib.sha256(Path(weights).read_bytes()).hexdigest()
print("Hub commit:", hub_commit)
print("weights sha256:", hub_sha256)
assert hub_sha256 == EXPECTED_SHA256, "the weights on the Hub are not the Stage 5 adapter"

## 3. Run the model card's usage example as is

The file is executed unchanged: it loads the base model in 4 bits, applies the adapter from the
Hub and classifies two example reviews. Its `classify` function is then used below.

In [ ]:
import time

card_code = Path("model_card/usage_example.py").read_text(encoding="utf-8")
card = {"__name__": "usage_example"}
start = time.perf_counter()
exec(compile(card_code, "model_card/usage_example.py", "exec"), card)
print(f"loaded and ran the usage example in {time.perf_counter() - start:.0f} s")

## 4. Selection set with the card's `classify`

One review at a time, 3 forward passes each (one per label).

In [ ]:
from ptbr_review_lora.metrics import compute_metrics
from ptbr_review_lora.predictions import load_predictions

start = time.perf_counter()
preds = [card["classify"](text) for text in select_texts]
SCORE_SECONDS = round(time.perf_counter() - start)

saved = load_predictions(Path("predictions/dev_select/smollm3_qlora_human.csv"), len(select))
metrics = compute_metrics(y_select, preds)
agreement = sum(p == s for p, s in zip(preds, saved)) / len(preds)
f1_diff = metrics["macro_f1"] - REFERENCE_F1
passed = abs(f1_diff) <= F1_TOLERANCE and agreement >= MIN_AGREEMENT

print(f"macro-F1 {metrics['macro_f1']:.4f} (Stage 5: {REFERENCE_F1:.4f}, difference {f1_diff:+.4f})")
print(f"agreement with the saved predictions: {agreement:.2%} ({round((1 - agreement) * len(preds))} flips)")
print(f"scoring time: {SCORE_SECONDS} s")
print("PASS" if passed else "FAIL")

## 5. Report and download

The report has numbers only (no review text).

In [ ]:
import json
import platform
import zipfile

import bitsandbytes
import peft
import transformers

report = {
    "adapter_repo": ADAPTER_REPO,
    "hub_commit": hub_commit,
    "weights_sha256": hub_sha256,
    "expected_sha256": EXPECTED_SHA256,
    "repo_commit": REPO_COMMIT,
    "environment": {
        "platform": PLATFORM, "gpu": GPU_NAME, "python": platform.python_version(),
        "torch": torch.__version__, "transformers": transformers.__version__,
        "peft": peft.__version__, "bitsandbytes": bitsandbytes.__version__,
    },
    "split": "repro_dev_select",
    "n": len(select),
    "criterion": {"reference_macro_f1": REFERENCE_F1, "f1_tolerance": F1_TOLERANCE, "min_agreement": MIN_AGREEMENT},
    "results": metrics,
    "macro_f1_difference": f1_diff,
    "agreement_with_saved": agreement,
    "score_seconds": SCORE_SECONDS,
    "passed": passed,
}
Path("reports/hub_check_select.json").write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

ZIP_PATH = f"{OUTPUT_DIR}/hub_check_results.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("reports/hub_check_select.json")
print("saved", ZIP_PATH)

if PLATFORM == "colab":
    from google.colab import files

    files.download(ZIP_PATH)
else:
    print("Kaggle: download it from the Output tab of this notebook version")
assert passed, "the published adapter did not reproduce the Stage 5 result (see above)"